In [1]:
import pandas as pd 
import duckdb
import os
import requests
from datetime import datetime
from dotenv import load_dotenv , find_dotenv
import math 

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')

# 1. 환경 변수 로드
load_dotenv(find_dotenv())
OPINET_API_KEY_GROUP = ['OPINET_API_KEY_1', 'OPINET_API_KEY_2', 'OPINET_API_KEY_3', 
                        'OPINET_API_KEY_4', 'OPINET_API_KEY_5', 'OPINET_API_KEY_6' ] 
MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT') 
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')


#2. duckdb를 통한 s3 읽기 설정
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"SET s3_endpoint='{MINIO_ENDPOINT}';")
con.execute(f"SET s3_access_key_id='{MINIO_ACCESS_KEY}';")
con.execute(f"SET s3_secret_access_key='{MINIO_SECRET_KEY}';")
con.execute("SET s3_url_style='path'; SET s3_use_ssl='false';")


print("✅ DuckDB의 MinIO 접속 준비 완료!")



✅ DuckDB의 MinIO 접속 준비 완료!


In [ ]:
# A1. 기초적인 Pearson Correlation 계산
df = con.sql(
    """
    with base as (
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    and part_dt >= '20260101'
    group by 1
    )
    select 
    CORR(gasoline_prc , disel_prc) as pearson_correlation
    from base 
"""
).df()

display(df)

,pearson_correlation
0,0.99153


In [5]:
# A2. 연도별 Pearson Correlation 계산
import matplotlib

df = con.sql(
    """
    with base as (
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    group by 1
    )
    select 
    substring( cast(part_dt AS STRING), 1,4) as yyyy
    , CORR(gasoline_prc , disel_prc) as pearson_correlation
    from base 
    group by 1
    order by 1
"""
).df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [16]:
# A1. 기초적인 Pearson Correlation 계산
df = con.sql(
    """
    select 
    part_dt
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as disel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    and part_dt >= '20200101'
    group by 1
    """
).df()

display(df)

,part_dt,gasoline_prc,disel_prc
0,20200229,1525.03,1344.88
1,20200104,1561.87,1393.82
2,20200326,1418.18,1224.52
3,20200214,1545.01,1370.38
4,20210313,1509.97,1309.59
...,...,...,...
2339,20260526,2011.22,2005.64
2340,20260527,2011.09,2005.31
2341,20260529,2010.81,2005.45
2342,20260530,2010.77,2005.26


In [ ]:
#A3. pandas로 corr 계산하기
import pandas as pd

corr = df['gasoline_prc'].corr(df['disel_prc'])
display(corr)

0.9925729779446998

In [ ]:
#A4. 연도별 corr 계산하기
import pandas as pd 
df['yyyy'] = df['part_dt'].astype(str).str[:4]

yearly_corr = df.groupby('yyyy').apply(lambda y: y['disel_prc'].corr(y['gasoline_prc'])).reset_index().rename(columns = {0:'corr'})

display(yearly_corr)

C:\Users\B450M\AppData\Local\Temp\ipykernel_7104\987361968.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  yearly_corr = df.groupby('yyyy').apply(lambda y: y['disel_prc'].corr(y['gasoline_prc'])).reset_index().rename(columns = {0:'corr'})


,yyyy,corr
0,2020,0.998265
1,2021,0.995719
2,2022,0.708412
3,2023,0.562906
4,2024,0.756459
5,2025,0.958608
6,2026,0.992573


: 

In [ ]:
# A3. Pearson계산을 직접 수식으로 진행하면서 Cosine과 얼마나 유사한지 체크하자.
df = con.sql(
    """
    with step1 as (
    select 
    part_dt
    , 1 as join_key
    , max(case when prodcd = 'B027' then price end ) as gasoline_prc
    , max(case when prodcd = 'D047' then price end ) as diesel_prc
    from read_parquet('s3://petroleum-project/national_avg/price/*/*.parquet')
    where 1=1
    and part_dt >= '20260101'
    group by 1,2
    )
    , type_avg as (
    select
    1 as join_key
    , avg ( gasoline_prc) as avg_gasoline_prc
    , avg( diesel_prc) as avg_diesel_prc
    from step1 
    group by 1
    )
    , diff as (
    select
    t1.part_dt
    , t1.gasoline_prc
    , t2.avg_gasoline_prc
    , t1.gasoline_prc -  t2.avg_gasoline_prc as diff_gasoline
    , t1.diesel_prc
    , t2.avg_diesel_prc
    , t1.diesel_prc - t2.avg_diesel_prc as diff_diesel
    , (t1.gasoline_prc -  t2.avg_gasoline_prc) * (t1.diesel_prc - t2.avg_diesel_prc) as numerator
    from step1 t1
    left join type_avg t2
    on t1.join_key = t2.join_key
     )
    ,denom as (
    select
    1 as join_key
    , sqrt(sum(diff_gasoline ** 2)) as gasoline_l2_norm
    , sqrt(sum(diff_diesel ** 2)) as diesel_l2_norm
    , sqrt(sum(diff_gasoline ** 2)) * sqrt(sum(diff_diesel ** 2)) as denom
    from diff 
    group by 1
    )
    , numerator as (
    select
    1 as join_key 
    , sum(numerator) as numerator_sum
    from diff 
    group by 1
    )
    , manual_cal as (
    select
    t1.join_key
    , t1.numerator_sum / t2.denom as pearson_r
    from numerator t1
    left join denom t2
    on t1.join_key = t2.join_key
    group by 1
    )
    , auto_cal as (
    select
    1 as join_key
    , corr(gasoline_prc , diesel_prc) as auto_cal 
    
    from step1
    group by 1
    )
    select 
    t1.join_key
    , t1.pearson_r as manual_pearson
    , t2.auto_cal as auto_pearson 
    from manual_cal t1
    left join auto_cal t2
    on t1.join_key = t2.join_key


    

    
    
"""
).df()

display(df)